# 03 — Policy Experiments: A Small CGE Laboratory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/03_policy_experiments.ipynb)

Here we solve **one benchmark equilibrium**, create several independent Scenario objects from it, and keep those scenarios alive at the same time:

1. abolish all import tariffs;
2. abolish all production taxes;
3. increase the capital endowment by 10%.

This is the key v0.6 workflow advantage: one protected benchmark can support several isolated counterfactual branches without manual resetting.

## 1. Setup and solve one benchmark

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab, set the environment variable CGE_CORE_REF to test a branch or tag.
# The public notebooks default to main. Outside Colab, a current CGE-Core git
# checkout is used directly, so branch development never silently resets to main.
CGE_CORE_REF = os.environ.get("CGE_CORE_REF", "main")
IN_COLAB = Path("/content").exists()

if not IN_COLAB and (Path.cwd() / ".git").is_dir() and (Path.cwd() / "cge_core").is_dir():
    REPO_DIR = Path.cwd()
    source_label = "current checkout"
else:
    WORKSPACE = Path("/content") if IN_COLAB else Path.home() / ".cache"
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = WORKSPACE / "CGE-core-colab"
    REPO_URL = "https://github.com/miraflor/CGE-core.git"

    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout.")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO_DIR)],
            check=True,
        )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", CGE_CORE_REF, "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
    source_label = CGE_CORE_REF

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("✓ CGE-Core", cge_core.__version__)
print("✓ Source:", source_label, f"({commit})")
print("✓ Repository:", REPO_DIR)


import shutil

if not shutil.which("ipopt"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "amplpy.modules", "install", "coin"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    module_path = subprocess.check_output(
        [sys.executable, "-m", "amplpy.modules", "path"],
        text=True,
    ).strip()
    os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)


In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from cge_core import CGE, example_data
from cge_core.models import StdCGE

GOODS = ["BRD", "MLK"]
model = CGE(model=StdCGE(), data=example_data("stdcge"))
benchmark = model.solve_benchmark(
    numeraire=("pf", "LAB"),
    redundant=("eqpf", "LAB"),
    solver=SOLVER,
)

print("✓ One benchmark equilibrium solved.")

## 2. Create independent scenarios from the same benchmark

In [ ]:
def equivalent_variation(benchmark, result):
    denom = math.prod(
        benchmark.value("alpha", good) ** benchmark.value("alpha", good)
        for good in GOODS
    )
    return result.objective / denom - benchmark.objective / denom

# These three Scenario objects coexist. None shares mutable scenario state
# with another, and none can mutate the protected benchmark snapshot.
tariff_scenario = benchmark.scenario("Abolish import tariffs")
production_tax_scenario = benchmark.scenario("Abolish production taxes")
capital_scenario = benchmark.scenario("Capital endowment +10%")

print("✓ Three independent Scenario objects are live at once.")

## 3. Apply three qualitatively different shocks and solve

In [ ]:
for good in GOODS:
    tariff_scenario.set("taum", good, 0.0)
    production_tax_scenario.set("tauz", good, 0.0)

capital0 = benchmark.value("FF", "CAP")
capital_scenario.set("FF", "CAP", 1.10 * capital0)

scenario_objects = [
    tariff_scenario,
    production_tax_scenario,
    capital_scenario,
]

scenarios = []
for scenario in scenario_objects:
    result = scenario.solve(solver=SOLVER)
    scenarios.append({
        "label": scenario.name,
        "scenario": scenario,
        "result": result,
        "results": result.compare(benchmark),
        "ev": equivalent_variation(benchmark, result),
    })

print("✓", len(scenarios), "independent scenarios solved")

## 4. Compare welfare

In [ ]:
welfare = pd.DataFrame(
    [{"scenario": s["label"], "equivalent_variation": s["ev"]} for s in scenarios]
)
display(welfare.style.format({"equivalent_variation": "{:+.4f}"}))

## 5. Compare output responses

In [ ]:
rows = []
for scenario in scenarios:
    part = scenario["results"]
    part = part[part["component"] == "Z"]
    for _, row in part.iterrows():
        rows.append({
            "scenario": scenario["label"],
            "good": row["index_1"],
            "pct_change": row["pct_change"],
        })

output_changes = pd.DataFrame(rows)
pivot = output_changes.pivot(index="good", columns="scenario", values="pct_change")
display(pivot)

ax = pivot.plot(kind="bar", figsize=(8, 4))
ax.axhline(0, linewidth=0.8)
ax.set_ylabel("% change from benchmark")
ax.set_title("Gross output across policy scenarios")
plt.show()

## 6. Compare imports, household demand, and prices

In [ ]:
for component, title in [
    ("M", "Imports"),
    ("Xp", "Household demand"),
    ("pq", "Composite prices"),
]:
    rows = []
    for scenario in scenarios:
        part = scenario["results"]
        part = part[part["component"] == component]
        for _, row in part.iterrows():
            rows.append({
                "scenario": scenario["label"],
                "item": row["index_1"],
                "pct_change": row["pct_change"],
            })
    table = pd.DataFrame(rows).pivot(index="item", columns="scenario", values="pct_change")
    print(title)
    display(table.style.format("{:+.2f}%"))

## 7. Make your own scenario 👇

A shock is a tuple `(component_name, index, new_value)`. Combine tuples for a policy package.

In [ ]:
# 👇 EDIT THIS POLICY PACKAGE
CUSTOM_SHOCKS = [
    ("taum", "BRD", 0.05),
]

custom_scenario = benchmark.scenario("My custom scenario")
for component, index, new_value in CUSTOM_SHOCKS:
    custom_scenario.set(component, index, new_value)

custom_result = custom_scenario.solve(solver=SOLVER)
custom = custom_result.compare(benchmark)

display(
    custom[
        custom["component"].isin(["Z", "M", "Xp", "pq"])
    ][["component", "index_1", "reference_value", "value", "pct_change"]]
    .style.format({
        "reference_value": "{:.4f}",
        "value": "{:.4f}",
        "pct_change": "{:+.2f}%",
    })
)
print(f"Equivalent variation: {equivalent_variation(benchmark, custom_result):+.4f}")

## What you learned

A **model** is the economic structure. A **scenario** specifies exogenous changes. A **closure** specifies which variables are fixed and which absorb adjustment.

## Next

Notebook 04 turns a single SAM CSV into model-ready CGE-Core data.

[Open Notebook 04 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/04_bring_your_own_sam.ipynb)